# 4.1 · 线性回归 / Linear Regression

> **课程定位 / Where this fits**
> **Part 4 第 1 课, 也是全部监督学习的"Hello World"**。0.2 节我们用 NumPy 顺手解过一次正规方程; 这一课**正式**讲透：OLS 几何、正规方程推导、梯度下降、统计性质、假设。**所有更复杂的模型都是它的变奏**。
> The Hello World of supervised learning. We formalize OLS: geometry, normal equation, gradient descent, statistical properties, assumptions.

> 📐 **符号约定**（[`NOTATION.md`](../NOTATION.md)）：$\mathbf{X}\in\mathbb{R}^{n\times d}$ 设计矩阵, $\mathbf{w}\in\mathbb{R}^d$ 权重, $\hat{y}=\mathbf{X}\mathbf{w}$。

> 💡 **面试相关 / Interview-relevant**
> - "推导线性回归的正规方程" ★★★★★（白板必考）
> - "线性回归的假设有哪些" ★★★★★
> - "正规方程 vs 梯度下降何时用哪个" ★★★★
> - "R² 的含义和局限" ★★★★

---

## 学习目标 / Learning Objectives
1. 从**三个视角**理解 OLS：最小化残差平方 / 投影 / 高斯 MLE。
2. **推导正规方程** $\hat{\mathbf{w}}=(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$ 并从零实现。
3. 用梯度下降求同一个解, 知道二者的取舍。
4. 解读系数、R²、统计显著性。
5. 列出并理解线性回归的**五大假设**。

## 目录 / TOC
1. [模型与三个视角 ⭐](#1)
2. [🏠 数据](#2)
3. [正规方程：推导 + 从零实现 ⭐](#3)
4. [梯度下降解法](#4)
5. [对照 sklearn + 系数解读](#5)
6. [评估：R² / RMSE / MAE](#6)
7. [统计推断：系数显著性 (statsmodels)](#7)
8. [五大假设 ⭐](#8)
9. [小结](#9)


<a id="1"></a>
## 1. 模型与三个视角 ⭐ / Model & Three Views

**模型**（含偏置, 把 1 列拼进 $\mathbf{X}$）：
$$\hat{y}_i = w_0 + w_1 x_{i1} + \dots + w_d x_{id} = \mathbf{x}_i^\top \mathbf{w}$$

**目标**：最小化均方误差 / 残差平方和：
$$J(\mathbf{w}) = \frac{1}{n}\sum_i (y_i - \mathbf{x}_i^\top\mathbf{w})^2 = \frac{1}{n}\|\mathbf{y} - \mathbf{X}\mathbf{w}\|_2^2$$

**同一件事的三个视角**（理解透彻 = 面试碾压）：

| 视角 | 说法 | 来自 |
|---|---|---|
| **代数/优化** | 最小化残差平方和 | 本课主线 |
| **几何/投影** | $\hat{\mathbf{y}}$ = $\mathbf{y}$ 到列空间 $\mathcal{C}(\mathbf{X})$ 的正交投影 | 0.7 线性代数 |
| **统计/MLE** | 高斯噪声假设下的极大似然 | 2.9 MLE |

**几何视角最深刻**：残差 $\mathbf{y}-\hat{\mathbf{y}}$ 垂直于列空间 → $\mathbf{X}^\top(\mathbf{y}-\mathbf{X}\mathbf{w})=\mathbf{0}$ → **直接得正规方程**。
The geometric view: residuals are orthogonal to the column space, which immediately yields the normal equation.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

from sklearn.datasets import fetch_california_housing
data = fetch_california_housing(as_frame=True)
X_df, y = data.data, data.target
print(f"California Housing: {X_df.shape}, target=房价中位数(10万$)")
print(X_df.columns.tolist())


<a id="3"></a>
## 3. 正规方程：推导 + 从零实现 ⭐ / Normal Equation

**推导**（最重要的白板题）。目标 $J(\mathbf{w}) = \|\mathbf{y}-\mathbf{X}\mathbf{w}\|^2$, 对 $\mathbf{w}$ 求梯度（0.8 节矩阵微积分）：

$$\nabla_{\mathbf{w}} J = -2\mathbf{X}^\top(\mathbf{y} - \mathbf{X}\mathbf{w})$$

令 $=\mathbf{0}$：
$$\mathbf{X}^\top\mathbf{X}\mathbf{w} = \mathbf{X}^\top\mathbf{y} \;\Rightarrow\; \boxed{\hat{\mathbf{w}} = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}}$$

⚠ **实现时永远用 `solve` 不用 `inv`**（0.2/0.7 节铁律：数值稳定 + 快）。


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = X_df.values
X_tr, X_te, y_tr, y_te = train_test_split(X, y.values, test_size=0.3, random_state=0)

# 标准化 (只 fit train, 3.4 防泄漏) + 加偏置列 / scale then add bias
scaler = StandardScaler().fit(X_tr)
Xtr_s, Xte_s = scaler.transform(X_tr), scaler.transform(X_te)
Xtr_b = np.c_[np.ones(len(Xtr_s)), Xtr_s]      # 拼 1 列做偏置 / bias column
Xte_b = np.c_[np.ones(len(Xte_s)), Xte_s]

# 正规方程: solve(XᵀX, Xᵀy), 不用 inv / normal equation via solve
def fit_ols(X, y):
    return np.linalg.solve(X.T @ X, X.T @ y)

w = fit_ols(Xtr_b, y_tr)
print(f"从零实现的权重 (含偏置 w0):")
print(f"  w0 (截距) = {w[0]:.4f}")
for name, wi in zip(data.feature_names, w[1:]):
    print(f"  {name:<12} {wi:+.4f}")


<a id="4"></a>
## 4. 梯度下降解法 / Gradient Descent

正规方程 $O(d^3)$（求逆/解方程）。$d$ 极大或数据流式时改用**梯度下降**（0.8/0.10 节）：
$$\mathbf{w} \leftarrow \mathbf{w} - \eta \cdot \frac{2}{n}\mathbf{X}^\top(\mathbf{X}\mathbf{w} - \mathbf{y})$$


In [ ]:
def fit_gd(X, y, eta=0.1, n_iter=500):
    n, d = X.shape
    w = np.zeros(d)
    losses = []
    for _ in range(n_iter):
        grad = (2/n) * X.T @ (X @ w - y)
        w -= eta * grad
        losses.append(np.mean((X @ w - y)**2))
    return w, losses

w_gd, losses = fit_gd(Xtr_b, y_tr)
print(f"梯度下降权重 vs 正规方程权重: 最大差异 = {np.abs(w_gd - w).max():.6f}")
print("→ 两种方法殊途同归 (凸问题, 唯一最优解)")

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(losses); ax.set_xlabel("iteration"); ax.set_ylabel("MSE"); ax.set_yscale("log")
ax.set_title("梯度下降收敛 (凸 → 稳定下降到全局最优)")
plt.tight_layout(); plt.show()


**正规方程 vs 梯度下降的取舍**：

| | 正规方程 | 梯度下降 |
|---|---|---|
| 复杂度 | $O(nd^2 + d^3)$ | $O(nd)$ 每步 |
| 适合 | $d$ 小（< 1万）| $d$ 大 / 数据流 / 不可整体载入 |
| 调参 | 无 | 要调 $\eta$ |
| 解 | 精确（一步）| 迭代逼近 |

→ **sklearn 的 `LinearRegression` 用的是 SVD-based 最小二乘（比正规方程更稳）, 不是梯度下降**。神经网络才用梯度下降。


<a id="5"></a>
## 5. 对照 sklearn + 系数解读 / sklearn & Coefficients


In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression().fit(Xtr_s, y_tr)    # sklearn 自带处理截距
print(f"sklearn 截距 = {lr.intercept_:.4f}  vs 我们的 w0 = {w[0]:.4f}")
print(f"系数最大差异 = {np.abs(lr.coef_ - w[1:]).max():.6f} → 完全一致\n")

# 系数解读 (因为标准化了, 系数大小 = 重要性可比) / interpret coefficients
coef = pd.Series(lr.coef_, index=data.feature_names).sort_values(key=abs, ascending=False)
print("系数 (标准化后, 可比较大小):")
print(coef.round(3))
print(f"\n解读: MedInc(收入) 系数 +{coef['MedInc']:.2f} = 收入每升高1个标准差, 房价中位数升 {coef['MedInc']:.2f}(10万$)")
print("Latitude/Longitude 负系数 = 越往北/西房价越低 (地理效应)")


<a id="6"></a>
## 6. 评估：R² / RMSE / MAE / Evaluation Metrics

| 指标 | 公式 | 含义 |
|---|---|---|
| **R²** | $1 - \frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}}$ | 解释了多少比例的方差; 1=完美, 0=和瞎猜均值一样 |
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y-\hat{y})^2}$ | 和 y 同单位, 对大误差敏感 |
| **MAE** | $\frac{1}{n}\sum\|y-\hat{y}\|$ | 同单位, 对异常值稳健 |

(完整指标体系 Part 5.1; 这里够用。)


In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

y_pred = lr.predict(Xte_s)
print(f"test R²   = {r2_score(y_te, y_pred):.4f}  (解释了 {r2_score(y_te, y_pred):.0%} 的房价方差)")
print(f"test RMSE = {np.sqrt(mean_squared_error(y_te, y_pred)):.4f} (10万$)")
print(f"test MAE  = {mean_absolute_error(y_te, y_pred):.4f} (10万$)")
print(f"\nMAE < RMSE → 存在一些大误差把 RMSE 拉高 (RMSE 惩罚大误差更重)")

# 预测 vs 真实图 / predicted vs actual
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y_te, y_pred, alpha=0.2, s=8)
ax.plot([0, 5], [0, 5], "r--", lw=2)
ax.set_xlabel("actual"); ax.set_ylabel("predicted")
ax.set_title("预测 vs 真实 (点越贴对角线越好)")
plt.tight_layout(); plt.show()
print("注意 y=5 处一条竖线 — target 被截顶在 5 (0.2 节就发现的数据陷阱), 线性模型无法预测>5")


<a id="7"></a>
## 7. 统计推断：系数显著性 / Statistical Inference

sklearn 只给点估计; **statsmodels 给完整统计报告**（系数标准误、t 值、p 值、置信区间——接 2.5/2.6/2.9）。系数的 SE 来自 2.9 节的 $(\mathbf{X}^\top\mathbf{X})^{-1}\sigma^2$。


In [ ]:
import statsmodels.api as sm

# statsmodels 要手动加常数项 / add constant manually
X_sm = sm.add_constant(Xtr_s)
ols = sm.OLS(y_tr, X_sm).fit()
print(ols.summary().tables[1])    # 系数表 / coefficient table
print("\n每个系数都有: std err(2.9), t值, P>|t|(2.6 假设检验), 95% CI(2.5)")
print("p < 0.05 → 该特征显著; 这就是把 Part 2 统计学用到回归上")


<a id="8"></a>
## 8. 五大假设 ⭐ / The Five Assumptions

线性回归的**统计推断**（p 值/CI 可信）依赖五大假设（记忆口诀 **LINE-N** 略有出入, 这里按常见五条）：

| 假设 | 含义 | 违反后果 | 检验/修复 (4.2 详述) |
|---|---|---|---|
| **1. 线性 Linearity** | y 与特征线性相关 | 系数有偏 | 残差图; 加多项式(4.3) |
| **2. 独立 Independence** | 残差互相独立 | SE 错 | 时序自相关检验 |
| **3. 同方差 Homoscedasticity** | 残差方差恒定 | SE 错 | 残差图; 稳健SE/WLS |
| **4. 正态 Normality (of residuals)** | 残差近似正态 | 小样本 CI 不准 | QQ 图(2.2) |
| **5. 无多重共线 No multicollinearity** | 特征间不高度相关 | 系数不稳/不可解释 | VIF(4.2) |

⚠ **常见误解**：假设是关于**残差**的, 不是关于 y 或 x 的分布。且**预测**对假设不那么敏感, **推断**（信 p 值/CI）才严格要求。
The assumptions are about the residuals, not y or x. Prediction is forgiving; inference is strict.

> 💡 第 1/3/4/5 条的检验全在 **4.2 回归诊断** 实操——本课先建立认知。


<a id="9"></a>
## 9. 小结 / Summary

```
模型: ŷ = Xw; 最小化 ‖y-Xw‖²
三视角: 残差最小 / 投影到列空间 / 高斯 MLE
正规方程: ŵ = (XᵀX)⁻¹Xᵀy ⭐ (实现用 solve 不用 inv)
梯度下降: w -= η·(2/n)Xᵀ(Xw-y); d 大/流式时用
评估: R²(解释方差比) / RMSE(惩罚大误差) / MAE(稳健)
推断: statsmodels 给 SE/t/p/CI (用 Part 2 统计)
五假设: 线性·独立·同方差·残差正态·无共线 (针对残差, 推断才严格)
```

### 💡 面试速查
1. **正规方程推导**: ∇J=−2Xᵀ(y−Xw)=0 → ŵ=(XᵀX)⁻¹Xᵀy, 白板能写
2. **正规方程 vs GD**: d 小用前者(精确一步), d 大/流式用 GD
3. **R²**: 解释方差比例; 但 R² 永随特征增加 → 用 adjusted R²(4.2)
4. **五大假设针对残差**; 预测宽容, 推断严格
5. **sklearn 用 SVD 最小二乘**, 不是梯度下降(那是 NN)

### 下一节
**4.2 回归诊断**——本课列了五大假设, 下一课**实操检验**它们: 残差图、异方差、VIF、影响点。
